# 📋 STR01 · 미션 06 | 군단/사단 관할 부대 보급 수준(DOS) 실시간 현황 지도 및 경보 대시보드 구현

**교관용 Solution · 외부 배포 금지**  
CMS 편성: 3시간 · 교육용 가상 데이터 사용

---

## 📖 1. 상황과 이번 미션의 문제

가상의 20개 조직과 5종 물품의 기록을 이용해 **군단/사단 관할 부대 보급 수준(DOS) 실시간 현황 지도 및 경보 대시보드 구현** 과제를 수행합니다.

실제 조직의 기록이 아닌 실습자료이며, 코드를 실행한 결과도 실제 현장 상태를 뜻하지 않습니다.

**이번 미션에서 해결할 문제:** 20개 부대를 격자형 가상 지도에 배치해 부대별 최악 경보 등급을 색으로 표시하고, 3개 시점의 갱신 결과를 출력한다.

경보 건마다 각 피처를 평균값으로 바꿨을 때 예측이 변하는 정도를 계산해 기여도 상위 3개 인자를 자동으로 설명하고, 설명이 실제로 예측에 영향을 주는지 점검한다.

**만들 결과물:** `src/explainability/str01_xai_explainer.py`, `reports/str01_xai_report.md`, `outputs/M06/results.json`, `forms/STR01_M06_forms.md`

## 📌 2. 처음 시작하는 방법

1. 아래 **환경 준비 셀**을 실행합니다. Colab에서는 메뉴의 **런타임 → 모두 실행** 대신 각 셀의 결과를 먼저 확인하면서 진행해도 됩니다.
2. 핵심 함수의 `TODO`를 AI와 함께 구현합니다. 처음에는 함수의 입력·출력과 왜 필요한지 질문하세요.
3. 첫 실행 결과를 확인하고 실패하면 오류 메시지를 AI에게 설명해 달라고 요청합니다.
4. AI가 제안한 코드 중 최소 3가지를 원자료와 실행 결과로 대조하고 수정 이유를 기록합니다.
5. 활동별 작성 칸을 채운 뒤 **실제 AI 대화 원문**을 보존하고 마지막 저장 셀로 ZIP을 만듭니다.

> **보안과 평가:** 실제 조직·위치·재고·작전 자료, 개인정보를 외부 AI에 입력하지 않습니다.

- 모델 성능은 **실제 출력 로그**로 판단하며, 예시값을 실측으로 적지 않습니다.
- 실행 성공이나 ZIP 저장은 학습 PASS·실제 승인과 다릅니다.
- 폐쇄망에서는 허용된 Python/Jupyter 환경에서 실행하세요.
- 인터넷이 필요한 `pip` 설치는 폐쇄망에서 실패할 수 있으므로 사전 패키지 준비가 필요합니다.

**실습 전 필수:** ① 환경 설정 셀만 실행하면 됩니다.

실습자료(미션 lesson·forms, 공통 안내)가 노트북에 내장되어 있어 ZIP 업로드 없이 자동으로 준비됩니다.

준비 완료 문구를 확인한 뒤 다음 셀로 진행하세요.


In [ ]:
# @title ① 실습자료 자동 준비 및 환경 설정 (ZIP 업로드 불필요) (실행 버튼을 누르세요) { display-mode: "form" }
# 이 셀은 코드 숨김 상태로 배포됩니다. 소스코드를 열지 않아도 실행할 수 있습니다.
from pathlib import Path
import os, sys, json, re, math, time, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# 업로드 파일명은 자유이며, .zip 확장자와 유효한 ZIP 내용만 확인합니다.
current_dir = Path.cwd().resolve()
notebook_directory = globals().get('notebook_directory', current_dir)
workspace_name = 'STR01_실습프로젝트_작업공간'
WORKSPACE = next(
    (folder for folder in (*reversed(current_dir.parents), current_dir) if folder.name == workspace_name),
    current_dir / workspace_name,
)
STR01_PROJECT_READY = False

def _prepare_project_zip(raw_bytes):
    global STR01_PROJECT_READY, STR01_PROJECT_DIR
    import io
    if not zipfile.is_zipfile(io.BytesIO(raw_bytes)):
        raise ValueError('유효한 ZIP 파일이 아닙니다. 압축파일 내용을 확인하세요.')
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw_bytes)) as archive:
        for member in archive.infolist():
            name = member.filename.replace('\\', '/')
            target = (WORKSPACE / name).resolve()
            if not target.is_relative_to(WORKSPACE):
                raise ValueError('안전하지 않은 압축 경로가 발견되어 중단했습니다: ' + name)
            # 심볼릭 링크는 작업공간 외부 접근 가능성을 고려해 거부합니다.
            if (member.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('심볼릭 링크가 포함된 ZIP은 허용하지 않습니다: ' + name)
        for member in archive.infolist():
            target = (WORKSPACE / member.filename.replace('\\', '/')).resolve()
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            if target.is_file():
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, target.open('xb') as destination:
                import shutil
                shutil.copyfileobj(source, destination)
    os.chdir(WORKSPACE)
    STR01_PROJECT_DIR = WORKSPACE
    STR01_PROJECT_READY = True
    print('✅ 실습자료 준비 완료 (기존 파일은 보존):', WORKSPACE)
    print('✅ 이제 다음 교관용 실습 코드셀을 실행할 수 있습니다.')

# 실습자료가 노트북 안에 내장되어 있어 ZIP 업로드 없이 자동으로 준비됩니다.
# 내장 파일 2개: STR01_M06_forms.md, STR01_M06_lesson.md
import base64 as _embedded_b64
_EMBEDDED_ZIP_B64 = (
    'UEsDBBQAAAAIAAAAIVxEVFN9UwgAALATAAASAAAAU1RSMDFfTTA2X2Zvcm1zLm1k3VjfaxtXFn7XX3EhLOyCf6XJ9sEvS2AfttCyS9OHQikh7XohS7oOtsO2'
    'IMo4HgXFUrCcaOyRMzKjxKqtrELGsmyNQemD+58kb3Pv/A/9zrn3zozsbEpL+1ISIun+POc73/nOubkkrn/y8dxl8dHc+0Ltbir3UKitTVVri9dOUyTHkawd'
    'zKp7fXyIZOikXijkiSPrjpBHwyR+IFTVV3vOH//69+t/Eqq2p2pBErki9d201RNq35EbrpDRhkgGr7BDYCeW4Jt8HOD4PhaWSupJQx7FIt14kETOvPhMuWG6'
    '1v588pSz0afXPhDK3ZPPK4KtU8/GagcfflXFHZEcDmXtVCRxpLb7dKtaW1WBK64kUSBUO1a7DXjYkBstoe5F6iSQ3VBcnpv7g2oPZS+aImfhDNZ0k+OGajns'
    'fX8I47ACvoXyaZR6fnIUCtWJ1KlPYOgd1WRQN6bRWnykLS/1AIUzUyrJIwfei+RorMJg3iJBoHYcmHc4VNsN5QbWYG2r7I4ZkFGYRI9UMJYdLMCdLd/YgFPT'
    'VhPY9tKtrmpj3/6qCjcFbsYJqdeTtaqs7c0I8u9ljGth/UPZCAgp1fZFMgrUacuYDeMyINk+WiGfH8iHceaYSwGTJ25+dql06ZINifEnGTjqu3apVDY8ORul'
    'j6ryee9sRJHZc1R7LMrG1SvaVcLaeFO2XmhXiVHp1gOYOk4OI5o9dMEMcyWOBsRHrijjtunpafGWfzEj8r/v/MW+MNXIIOOVHDRhKLkD2qqwQTZkfEPwz0YT'
    '8JpRWnUcKK9ig51u9FPvwMQGs0RlIAmAcUDGxnf7Aa5OGn8ZeXtu4L3zDuEe4J7uPAT+USCjAJ8cIRqWHQRqmiKnQuRsvUFZblWA3Z0Xr7c6dIisHSNEOukj'
    'Qctpwoyq7aHcrYv0iWt249BrH4DVQu53khgM2q5QJhGs7aEKwnm6NUTsjjmHwPZndQDxqglVwGcI6F4xI580wHa53iR4MUD7oBREG0pJBGJtFRwws3y+54JB'
    'zNzukDbCudR7Mc+ErOuMed5T4SqFCHIVIqt2mmrwgu6S/VhIv8HbQgdCIBB7MDFfax1+51prIRKo14Qlv5ifJW3a2Uhfq9pOZjMbQClJqdoZFtOdURRGdIyJ'
    'NuaME0mLjHzlM3Z0QMs1KJBuamIUk7wscuW1PutbtFITPjaGnLDIzEpdre+l63GeuzksmmkcYKYD7Q9C5B72o5yEHjxGhejaPHpaMfZiZx0CgmUG6/8P7Nvg'
    'vYDwWwdKJMlsGUSKEEbJ6kRFhNlXSEM19WMCU0Xfa5DXiPPJKIZt8Absb5BSpx5XJnlvqHZ6JBf3fBtHqBfS38JWiA8YqtaPCxHQ8WSyIz7rXUBcr8v2GNqT'
    'uhpeDjilpk0kOXDpZzbHCXp+Dv/yeNGCAu81/CbONly/Auystrru68LA15rbIAAnAXmtWwIg+AJZi41mafQIDhj5xBB1HFmKag8wnvGu1ockYcPLGF8JtwYo'
    'Bvmt0tFlmwC/CZcuuj3NTg6rWSDMT8qwYVV5VZPRJgSslDZklRgJJU9WOT3ALDQN5Gcsv4O/aBDw1WBX2A0CqTbY+jJWFd80EtL1SZbhPydbLq7ofNDlkVyk'
    '90Oq0ulaoGsAsoCUn78WFYNH1PopUvuN05Q1ojoShOriGDXExuGN43HIU+9Y1fqm88tqEALtHZi6uVuRIdhNJTHs5hVTDf4nrLhokRAF2ubaGwXJoE9povw9'
    'KEi22NLhnVK88PWd2zdv/Qcj/Aet28vYmEoX1Q9Y+txD5F9ORS5FnitrrUwgyFb3EOXN2MfcPmkS34qpyHMsMacukD/XV6WtAM2qPKLOzmW/z1XuslkiyhYm'
    'bVMhF9jpXC5NRJk/P8F5aifwnd4FECk0EPOT/T/pOJp+06Nzcwg2ra0WWijb6K494HZPMw8Xb/epdScoi68BUj2NtW5wp8x+uU+QmK6L9ZU7fA5x1inr4oQx'
    'Qo8WpegDdpq8yBBK6wTV0gPAQW1XH3JMVlM0dbtDcTIcgH/YaVhIMu294ibbXGRacFPRdAely9rZCK0P8N51bcA0fgyRLq1bVTiPi/9iaFYW71moGYB5sbz0'
    '5awx5OYXt27fWvlmdnllae7yja9v3rphJhaWZu58Q2Ewv4GD7kS0ddpm5OTS4n/F6/uPDIIgLLNRN+D8NmKAIBDjJMLzyJ3J3x0coMGQ3x666zV9ugqrsv6Y'
    'yZ+zl8EX1/927R+U/TGrcC7K1B3zWyq2VEAgQrySuFQqbx2Y0AFQChyqTnqqc2iaP+STTicrAFrrIOWbwazWP2wxnVAOLhITafbhwvLyIoeTE1DnH93Almlt'
    'BWNMQ3QxNFdsaDiWv7fXqVGKLDnsm6QTwWItnqc+4NZF017AJWtWK2umYoYGOYITgkBxvPrD9p+BY4/efOxW1s6yPpE8TgopAPgWjxoqJeD5FfMEU7t9tdYm'
    '2k6+cDFO1s2REupHJ3fMxYcYSwO/xYr9nAmFDUOGv0aDGbi1qbGkG8iWc2/lIQvWxFOQ79YWQThVFJkHt33LGyHJrNBJZx6+hSt0xnF/efxC3a/niWowLCac'
    'SHeor76I5AS3mFi6lDFfAZbar5qqz68D0uUwT828FeU+Bcze7c7qPOD/v2i7utJQXvIhtsLALgSLGfKW1LpqUgsL0VD/DNWbEksLdxaXVpYL83pk5qt/TonF'
    'uyt37mISZ88uLSzfvb2yPPNviMCU+Nfi0lfLs/xfXjcwe4N/Yw/Jg9YDfr/qruxpRTea/GzU6qKJimn2XbfDRPPtCEGXDd96aHtN6rpQtfhFg/eaG9CAvUJf'
    'yK2T7q5YwT1Kzt0qpA10vwDaj1BLAwQUAAAACAAAACFcsKikiMYJAACRGQAAEwAAAFNUUjAxX00wNl9sZXNzb24ubWTtWFFPG9kVfudXXCmq1ErExkm6anlZ'
    'Rd1VG7VRo80+VK0qYDe0omJLBKTdfakMDFkHk2IST7BZ2x3vQoCsVzsYg41kWon9J5u3uXf+Q79zzr3jMWSj5nFXfcF4PHPvOd/5zne+O1fU3fffG8up22Nv'
    'qW/zZRUdh7q4nzXLLXyoqJOP/UDpk7xezyt91Il6j5QpVMxO/sfv/PbuT5Qp7phiLQo9FVe8uHqgzF5eb3hKhxsqap/hCYUncQv+009rWL6FG0dGrlxRZmUJ'
    'D0RHfaVfHMSbtZGRKMzjovr1nVvY0TNeXeliRxdPTaOkmzXj1Wwg+shTptGKjoLzrnm4rl/sn3dlDYWgTDWPrcPYf2wj0Lt9u17sH+hiQRd3MsOZmXqFnrQ5'
    'mLqnbCiSjazIyZiVR7huan0ERHiYrZYyX/T1WnkoT9pSUo39CuIcdaHs0e7KVAqm1zRbJRX1QiwR+zVl6j3KEw8ab0e/WMU1ClSg+qykj3oEfPzskdLtsqlx'
    'UPHGIwQ6MnI1ucML4hXA9nXPrFYyygSeaXe4SGvH2A4QUqSyzgCM5HnAuVJQv5xZVPE/e8Y7VNeluC/zZVqj14zCJ6aOBX1PFysv835qZ15zXF0bi8KkToRC'
    '+wBpxRX/AqTIXYehOa3Efidd1uOa8VcT6jw9BOGoHoS7gI4qIygH63XaDd9NUKIiRuGhKQZUfvCKsTyp6WDXYumWjQ5dIaJwScXlvmlX6GaCsxtQjra+YTn6'
    'zz5tr/11VzSkr4/AnQojGfhIhvM88sxySMlISakJmCmejXFQXnxsVO0WrtaSjXxjhAF3UOM7Eq6YylL8bJfRYMYBSQRQitp5x5Wr6g/CgD8Ot+N593c3b9nV'
    'Fbc5VjDbZbs4A1I8fW3kNmyFJAnT3UDlxsZ+hFD1QThKqgEC4J7d6LjE7QIZaXXADUkm0J+HkqQyTal6oOSJQtReTyVOaFR9JAS6ZFL0km1b/XG1MP9hdvrj'
    '+7NTM3+d+mBmdmbxk+zC4vxYbuLjqZkJ+8P0fOb+JyxClLh+3oo3WvFaT5opqEVHx5zPF+vUO/gWP6QLm0R54xf0cue8+4u52akPqA7Ucu6idJLSqz2spk+W'
    '3JW47JFCBXVcVr+/dYcjPytDCcbV5P8c8GRGxf6xKbbwsQ90BiqCfYI80HUYtr90kcWfeermLYLcnJSBGZETNB9qbv04BLSxF46roUAn700tTmXfe/fmO7ff'
    'zXx0b5KFcr+PrVD0M6Rll6AyNnZ/TmSI0flPa5Dbsm9Wy9orANtcbuiXF/v6MQrWWEXj5d7CTwheQkOsaL+f4ZLNDKo8rEJo5MamMs82TbEO5Hg0TWA0Tfxp'
    'bv6jBYqRi3pv7sOF7Pz0/bn5xQVqe9K7qsfVkyLGRZBHqg2GtgtWEelei7AbRFdVvL5u6v03qhM9xRXCU/aHSSaSZ/yzcTU/93f17cMntrugJzTkpKWkoxgb'
    'xNSPQjSVlxkoD0+RdmfQ+Cw2VVKagl5/yvoLOTr10GdO1e7+6uYdYFzvYRshZFBzE5A7sOf0rLgPwQKHWDn8NWiIY0hY0eiJhCiIts44Cg9Z6aRrmNSUgJ2j'
    '7jqyK1TOu8IC0kVgzWoX9tDxpIu3cwT/36ZmZ0C76YnZ6Xt/np5PD8oUEYaKxBxnynuHvBczn3HAAG52bFOQejU7w22AkvRJRykQO/Q6NBVpRQdiYgksciQY'
    '64Q5uHzUA3oq6vajw1CR0D4ky1HZ0bsVtkGBTwQztYBbZ3k/Ol1ySRD5Yr9KchFXa5BOunLF/p8Tr3XkxduPmRzLLUQFKpZQo5ERK3riac67yIw1kLHH/qxc'
    'nFP9jOVAFJpm0nJqkKWm2L97XEOrstQEqJrHrgUqvfUpGiA6XX+bn+UYUrBAXWhY7zWjXs96DxcDyRLm9RnqXvUJKR0e6LbPI4qIdpACw6V+jVO3ksUZ2VrB'
    '/6GCO+4q58eVdvVNhrorO558smOXwrPAkq0bk4YHZrBktg9SqQyRYjzxdPC6zTyG3WFHPJIbf4OZfdEZ0ASrVmxgtkORuxvPe0sm2FSCSZpg0KKve7Y1dalG'
    'c5ddS7dmTqu2PEwHO5Y5PhY4K6q2gN5l8UwhfF3IJaQlPcHEJbAbLcET/LLUHQA9NF3cjZdDF9etxqjW8ZMCCMpuIbF0Ys50iSQDXfi2SmwHqCLFFoNCYHPr'
    'AGlCzNGJ9+UzAcedINyBvkO/joE1Hy/42HEJ4TQKN8TnwE9+vmpJLZ1mjUTSZWKilZCPep45YkUi6blXaEMHoZGjZvxqWDWNF4zhv2DViy2I1iiPH5i4wMe/'
    'rR5TjAcVRQQRwmFEmbVTQpVORStD87EznATr3iuUpoPzDaSinTfP62zv6qhxKTlUSY2o4298s/VTa8vFvjnho0PFaYWwYH9fdbnA6P0jN3YNGkFUvC5um1mC'
    'IZGaBN9FEVPPD/t6ZgoMmZ+WGWc5nd1MfCZAIPqTfsEXsGekHSiWC33SEROd7jLeWyICyCYMbbNdsN9JFNLotr9TW0iPUxDR8Vc4cw6Gt8UwPYRVvE1cu4zk'
    'kIdmAz2QZQLL7BXslGGZowNlMBjXA0PmfCxWYU5XPXSPZTBZmjezMpPWSaVukSvkt+j3uQeL9x/gdxgxuK6FB7OLC5m/LMyx52FjZtmMbhpNTVEcMEz7KwJl'
    'E507yhzoFPA9rT6jzsiiHFEXJ1D80c/Jx6SndPokIJ0lydKRUeYCKZwwheclDq34BPWp5ciNeIeYBraFrDbSotJ8XGjnmwj+VBtiC2ockIXGcHPoHUJY45Zl'
    '+FkGuIcvHlRfUT7rTbmv2WRUrGpRQnaWunLmMvxyRszC+PfmxQU3ue+BBHwTW95BaUjI2EfA+GwfUNR6bdce5InSlqMkYo1VOcRbe83jInVqVmLS6E42UVLi'
    '8y5sAfSs4TlNFPwYIqnjs4IdUiPXLMKc9xscMAl9+x3pp2eohPoyX/6eHQVe5sm8mZMD0zx0rzjaX4rvlj5F6Wyjgt9ZaVY8Yi1ZgimpMNr3N9ML0AmqDbty'
    'ma20A0dmDxNbJRw/+OVGUpHrtiJcufEf2FsNKzVJK1jVwhKIWOTxtAKUwWSpEW/A4pkV7UxOQLb6A9j/P+1/eNOeZymY3djNDswzii9TjNrRHkzFJtPhzBn4'
    'QUfdkI6SWfUGGjeqXuMORtV3WYNRMQbZy29wSAyk+9kzIFqegpAzJMDeWrRE+ImfOWU5ZRO7t0LUOnXCQPXhsvGss9j4ly02XXBbyIY8nbEHDQDSa596slGw'
    'g3qA1UjqbfhrXoZnZUF5J568Es/YFxFyyES28i5DL3cw6OQiiamc4ddwmg5Q8NQR5r9QSwECFAMUAAAACAAAACFcRFRTfVMIAACwEwAAEgAAAAAAAAAAAAAA'
    'pAEAAAAAU1RSMDFfTTA2X2Zvcm1zLm1kUEsBAhQDFAAAAAgAAAAhXLCopIjGCQAAkRkAABMAAAAAAAAAAAAAAKQBgwgAAFNUUjAxX00wNl9sZXNzb24ubWRQ'
    'SwUGAAAAAAIAAgCBAAAAehIAAAAA'
)
_EMBEDDED_ZIP_BYTES = _embedded_b64.b64decode(_EMBEDDED_ZIP_B64)
print('📦 노트북에 내장된 실습자료를 준비합니다. ZIP 업로드는 필요하지 않습니다.')
_prepare_project_zip(_EMBEDDED_ZIP_BYTES)


## 미션 핵심 개념과 원본 실습 설명

---

## 🎯 미션 06. 군단/사단 관할 부대 보급 수준(DOS) 실시간 현황 지도 및 경보 대시보드 구현

**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침

#### 미션 배경 및 당위성

- 관할 부대 보급 수준(DOS) 실시간 현황 지도 및 경보 대시보드를 구현한다.
- 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **관할 부대의 DOS 현황을 가상 지도와 경보 색상으로 보여 주는 대시보드를 구현하고, 경보마다 예측에 기여한 인자를 설명한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.
- 이 문제를 해결하기 위해 본 미션에서는 **20개 부대를 격자형 가상 지도에 배치해 부대별 최악 경보 등급을 색으로 표시하고, 3개 시점의 갱신 결과를 출력 수행하고 검증하는** 실습을 체계적으로 수행합니다.
- 학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/explainability/str01_xai_explainer.py 및 XAI 리포트` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)

#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군수 장교/군무원이 신뢰하고 조치할 수 있는 설명의 명확성

#### 미션 PASS 조건

- 정량 기준: 주요 예측 건당 기여도 상위 3개 인자 자동 산출률 100%

- 진행 기준: SHAP summary plot 및 waterfall plot 생성 파이프라인 완성

- 검증 방법: 실무자 대상 설명 근거 유용성 평가 검증

#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.


## 교관용 검증 관점

### AI 활용 예시 · 미션 06 전용

> 나는 Python을 처음 배우는 학습자야. **군단/사단 관할 부대 보급 수준(DOS) 실시간 현황 지도 및 경보 대시보드 구현**을 진행하고 있어. 다음 요구사항을 보고 핵심 함수의 입력·출력, 필요한 검증 3개를 쉬운 말로 설명해줘. `TODO` 구간만 구현할 수 있는 코드를 제안하고, 데이터를 미리 보지 않는 검증 방법과 경계 사례도 알려줘. 실행 결과는 만들지 말고 내가 실제 실행한 로그를 받은 다음 분석해줘.  
> 요구사항: src/explainability/str01_xai_explainer.py / explain: row → 예측값과 상위3개 입력 민감도. 평균값 대체로 예측 변화 절댓값을 비교한다. SHAP·인과·실제 결핍 원인으로 단정하지 않는다.

**AI 답변을 받으면:** 그대로 복사하지 말고 입력 시점·자료 범위·실제 실행 여부를 직접 확인하세요. 확인되지 않은 수치는 `미측정`, 수행하지 않은 활동은 `미실시`로 남깁니다.

원본 Solution 코드를 실행해 출력값과 조건을 대조하세요. 이 파일은 학생용에 배포하지 않습니다.

In [ ]:
from pathlib import Path

if not (globals().get('STR01_PROJECT_READY', False) and globals().get('STR01_PROJECT_DIR') and Path(STR01_PROJECT_DIR).is_dir()):
    print('먼저 ① 실습자료 자동 준비 및 환경 설정 셀을 실행한 뒤 이 셀을 다시 실행하세요. ZIP 업로드는 필요 없습니다.')
else:
    """20개 가상 부대를 격자형 가상 지도에 배치해 보급 수준(DOS) 현황과 3단계 경보를 색으로 표시하는 대시보드를 만들고, 경보 건마다 예측에 기여한 상위 3개 인자를 자동으로 설명(XAI)하며 그래프와 텍스트 보고를 출력한다."""
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.ensemble import GradientBoostingRegressor

    # ---------------------------------------------------------------------------
    # [교육용가상자료] 가상 부대 20개 x 핵심 보급품 5종 일일 재고·소모 시뮬레이터 (실제 부대·수량·기준일수 아님)
    # 핵심성과지표(KPI) = 기준 보급일수(DOS) = 현재 재고 / 최근 평균 일일 소모량. 목표 DOS는 교육용 가정값이다.
    # 결핍 위험 일수 H = 보급이 없다고 가정할 때 현재 재고가 바닥나기까지 남은 일수(미래 실제 소모량으로 계산한 정답).
    # ---------------------------------------------------------------------------
    ITEMS = {"식량": (100, 0.3, 10), "피복": (20, 0.1, 15), "유류": (300, 1.2, 7), "탄약": (60, 1.5, 6), "수리부속": (15, 0.5, 12)}   # (기본 일소모, 템포 민감도, 목표 DOS)
    N_UNIT, DAYS = 20, 160
    UNITS = [f"가상{u:02d}부대" for u in range(1, N_UNIT + 1)]


    def make_data(seed=1, n_unit=N_UNIT, days=DAYS):
        """일 단위 재고 원장: unit, item, day, tempo(실제 작전 템포), plan(사전 계획 템포), wx(악천후), consume, resupply, stock."""
        rng = np.random.default_rng(seed)
        rows = []
        for u in range(1, n_unit + 1):
            s, ph = rng.uniform(0.7, 1.5), rng.uniform(0, 6.28)
            surge = np.zeros(days)
            for st in rng.choice(days - 10, 4, replace=False):
                surge[st:st + int(rng.integers(5, 10))] = rng.uniform(0.4, 0.8)
            tempo = 1 + 0.25 * np.sin(2 * np.pi * np.arange(days) / 30 + ph) + surge
            plan = tempo + rng.normal(0, 0.08, days)
            wx = (rng.random(days) < 0.15).astype(int)
            for it, (b, sens, tg) in ITEMS.items():
                stock, off = tg * b * s * rng.uniform(0.8, 1.2), int(rng.integers(0, 4))
                for t in range(days):
                    c = b * s * (1 + sens * (tempo[t] - 1)) * (1 + 0.12 * wx[t]) * rng.normal(1, 0.08)
                    r = max(tg * b * s - stock, 0) * rng.uniform(0.9, 1.1) * (0.5 if rng.random() < 0.08 else 1) if (t % 4 == off and t > 0) else 0.0   # 4일 주기 정기 보급: 목표 재고까지 채움(가끔 수송 차질로 절반)
                    stock = max(stock + r - c, 0.0)
                    rows.append((f"가상{u:02d}부대", it, t, tempo[t], plan[t], wx[t], c, r, stock))
        return pd.DataFrame(rows, columns=["unit", "item", "day", "tempo", "plan", "wx", "consume", "resupply", "stock"])


    def make_samples(d, lo=21, hi=125):
        """(부대, 품목, 기준일 t0)별 특성(피처)과 정답 H. 피처는 t0 시점까지 알 수 있는 값과 사전 계획·기상 예보만 사용한다."""
        out = []
        for (u, it), g in d.groupby(["unit", "item"]):
            c, stk, rs = g["consume"].values, g["stock"].values, g["resupply"].values
            pl, wx, tp = g["plan"].values, g["wx"].values, g["tempo"].values
            cs = np.cumsum(c)
            tg = ITEMS[it][2] * ITEMS[it][0]
            for t0 in range(lo, hi):
                idx = np.searchsorted(cs, cs[t0] + stk[t0])
                if idx >= len(cs) or idx <= t0:
                    continue
                c3, c7, c14 = c[t0 - 2:t0 + 1].mean(), c[t0 - 6:t0 + 1].mean(), c[t0 - 13:t0 + 1].mean()
                last = [k for k in range(t0, t0 - 15, -1) if rs[k] > 0]
                out.append({"unit": u, "item": it, "t0": t0, "H": idx - t0, "stock": stk[t0], "c3": c3, "c7": c7, "c14": c14,
                            "dos7": stk[t0] / c7, "dos3": stk[t0] / c3, "trend": c7 / c14, "burst": c3 / c7,
                            "plan7": pl[t0 + 1:t0 + 8].mean(), "plan_ratio": pl[t0 + 1:t0 + 8].mean() / tp[t0], "tempo0": tp[t0],
                            "wx_fc": wx[t0 + 1:t0 + 8].mean(), "since_rs": (t0 - last[0]) if last else 15,
                            "rs_avg": rs[t0 - 13:t0 + 1].sum() / 14, "fill": stk[t0] / tg, "item_code": list(ITEMS).index(it)})
        return pd.DataFrame(out)


    def mape(a, b):
        return float(np.mean(np.abs(a - b) / a))

    d = make_data()
    s = make_samples(d)
    X = [c for c in s.columns if c not in ("unit", "item", "t0", "H")]
    tr = s[s["t0"] <= 90]
    gbr = GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.08, random_state=0).fit(tr[X], tr["H"])
    mean_x = tr[X].mean()
    LABEL = {"stock": "현재 재고", "c3": "최근 3일 소모", "c7": "최근 7일 소모", "c14": "최근 14일 소모", "dos7": "7일 기준 DOS", "dos3": "3일 기준 DOS", "trend": "소모 추세",
             "burst": "소모 급증도", "plan7": "다음 7일 계획 템포", "plan_ratio": "계획/현재 템포 비", "tempo0": "현재 작전 템포", "wx_fc": "악천후 예보", "since_rs": "보급 후 경과일",
             "rs_avg": "최근 보급량", "fill": "목표 대비 재고", "item_code": "품목"}
    def level(h):
        return "심각" if h <= 2.5 else "경계" if h <= 4.5 else "주의" if h <= 7.5 else "정상"
    ORDER = {"정상": 0, "주의": 1, "경계": 2, "심각": 3}
    COL = {"정상": "tab:green", "주의": "gold", "경계": "orange", "심각": "red"}

    def explain(row):
        """XAI: 각 피처를 학습 평균값으로 바꿨을 때 예측이 변하는 정도(기여도)로 상위 3개 인자를 뽑는다. 음수면 그 인자 때문에 결핍 일수가 짧아졌다는 뜻."""
        x = row[X].astype(float)
        alts = pd.DataFrame([x] * (len(X) + 1)).reset_index(drop=True)
        for i, f in enumerate(X):
            alts.loc[i + 1, f] = mean_x[f]
        p = gbr.predict(alts)
        contrib = {f: float(p[0] - p[i + 1]) for i, f in enumerate(X)}
        return float(p[0]), sorted(contrib.items(), key=lambda kv: -abs(kv[1]))[:3]

    # 가상 지도: 4행 x 5열 격자(실제 위치 아님)
    POS = {u: (i % 5, i // 5) for i, u in enumerate(UNITS)}
    print("[지도 대시보드] 가상 격자 지도(위치는 실제와 무관) [교육용가상자료]")
    snaps = {}
    for day in (100, 101, 102):
        now = s[s["t0"] == day].copy()
        now["pred"] = np.clip(gbr.predict(now[X]), 1, None)
        now["lv"] = now["pred"].map(level)
        worst = now.loc[now.groupby("unit")["pred"].idxmin()].set_index("unit")
        snaps[day] = (now, worst)
        cnt = now["lv"].value_counts().reindex(["심각", "경계", "주의", "정상"], fill_value=0).to_dict()
        print(f" {day}일차: 경보 분포 {cnt}, 가장 위험한 부대 {worst['pred'].idxmin()}({worst['item'][worst['pred'].idxmin()]}, 예상 결핍 {worst['pred'].min():.1f}일)")

    now, worst = snaps[102]
    alerts = now[now["lv"] != "정상"].sort_values("pred")
    print(f"\n[경보 설명(XAI)] 102일차 경보 {len(alerts)}건 중 상위 6건 (기여도 상위 3개 인자)")
    ok = 0
    rows = []
    for _, r in alerts.iterrows():
        base, top = explain(r)
        ok += (len(top) == 3 and all(np.isfinite(v) for _, v in top))
        rows.append((r["unit"], r["item"], r["lv"], base, top))
    for u, it, lv, base, top in rows[:6]:
        txt = ", ".join(f"{LABEL[f]}({v:+.1f}일)" for f, v in top)
        print(f" - {u} {it} [{lv}] 예상 결핍 {base:.1f}일 <- {txt}")
    print(f"설명 자동 산출률 {ok}/{len(alerts)} = {ok / max(len(alerts), 1):.0%}")
    fid = []
    for u, it, lv, base, top in rows[:20]:
        r = alerts[(alerts["unit"] == u) & (alerts["item"] == it)].iloc[0]
        k = [f for f in X if f not in [t for t, _ in top]][0]
        alt = pd.DataFrame([r[X].astype(float)] * 2).reset_index(drop=True)
        alt.loc[0, top[0][0]] = mean_x[top[0][0]]; alt.loc[1, k] = mean_x[k]
        p = gbr.predict(alt)
        fid.append(abs(base - p[0]) >= abs(base - p[1]))
    print(f"설명 타당성 점검: 1순위 인자를 제거할 때의 예측 변화가 무관한 인자 제거보다 큰 비율 {np.mean(fid):.0%}")

    fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
    for u, (x, y) in POS.items():
        lv = worst.loc[u, "lv"]
        ax[0].scatter(x, -y, s=1500, color=COL[lv], edgecolor="black"); ax[0].text(x, -y, u[2:4], ha="center", va="center", fontsize=9)
    ax[0].set_title("Virtual map: worst alert level per unit (102nd day)"); ax[0].axis("off")
    if rows:
        u, it, lv, base, top = rows[0]
        ax[1].barh([f for f, _ in top][::-1], [v for _, v in top][::-1], color="tab:blue")
        ax[1].set_title("Top-3 contributions (days) for the most urgent alert"); ax[1].set_xlabel("change in predicted days to stockout")
    plt.tight_layout(); plt.show()
    print(f"\n목표(주요 예측 건당 기여도 상위 3개 인자 자동 산출률 100%): {'달성' if ok == len(alerts) else '미달'}")


## 원본 교관용 예상 출력 — 실측이 아닌 참고 예시

<details>
<summary><b>💡 [참고] 미션 06 예상 실행 결과 (Output)</b></summary>

```text
[지도 대시보드] 가상 격자 지도(위치는 실제와 무관) [교육용가상자료]
 100일차: 경보 분포 {'심각': 8, '경계': 11, '주의': 29, '정상': 47}, 가장 위험한 부대 가상15부대(탄약, 예상 결핍 1.0일)
 101일차: 경보 분포 {'심각': 8, '경계': 14, '주의': 24, '정상': 51}, 가장 위험한 부대 가상20부대(탄약, 예상 결핍 1.0일)
 102일차: 경보 분포 {'심각': 9, '경계': 12, '주의': 24, '정상': 54}, 가장 위험한 부대 가상17부대(탄약, 예상 결핍 1.0일)

[경보 설명(XAI)] 102일차 경보 45건 중 상위 6건 (기여도 상위 3개 인자)
 - 가상17부대 탄약 [심각] 예상 결핍 0.9일 <- 3일 기준 DOS(-5.1일), 목표 대비 재고(-0.3일), 7일 기준 DOS(-0.2일)
 - 가상07부대 탄약 [심각] 예상 결핍 1.0일 <- 3일 기준 DOS(-5.2일), 7일 기준 DOS(-0.4일), 목표 대비 재고(-0.4일)
 - 가상03부대 유류 [심각] 예상 결핍 1.0일 <- 3일 기준 DOS(-5.4일), 목표 대비 재고(-0.4일), 7일 기준 DOS(-0.3일)
 - 가상18부대 탄약 [심각] 예상 결핍 1.1일 <- 3일 기준 DOS(-5.4일), 7일 기준 DOS(-0.3일), 목표 대비 재고(-0.3일)
 - 가상08부대 유류 [심각] 예상 결핍 1.2일 <- 3일 기준 DOS(-5.1일), 7일 기준 DOS(-0.4일), 목표 대비 재고(-0.3일)
 - 가상18부대 유류 [심각] 예상 결핍 1.3일 <- 3일 기준 DOS(-5.6일), 목표 대비 재고(-0.3일), 7일 기준 DOS(-0.3일)
설명 자동 산출률 45/45 = 100%
설명 타당성 점검: 1순위 인자를 제거할 때의 예측 변화가 무관한 인자 제거보다 큰 비율 100%

목표(주요 예측 건당 기여도 상위 3개 인자 자동 산출률 100%): 달성
```
</details>


## 📋 3. 학습활동 4단계

### 활동 1: 무엇을 확인하고 계획하나요?

**이번 활동의 과제:**

#### 1. 단계 개요

- **목표:**
  - 원본 목표, 기본 코드, 그리고 제공된 학습 자료를 면밀히 분석하여, "인자 하나를 평균값으로 바꾸는 설명의 한계는 무엇일까?"라는 핵심 질문에 대해 사전에 엄밀히 판단합니다.
  - 아울러 관할 부대 보급 수준(DOS) 현황을 가상 지도(4×5 격자)와 경보 색상으로 시각화하고 경보마다 예측 기여 인자를 산출하는 과정에서 AI 에이전트에 위임할 분석 범위, 데이터 제약 사항, 입력 및 기간·단위·사용 가능 시점, 그리고 explain 함수의 입출력 계약 조건을 체계적으로 수립합니다.
- **수행 방식:**
  - 막연한 코딩이나 AI의 무분별한 답변에 의존하지 않고, 군수 의사결정의 투명성과 XAI(설명 가능한 AI) 해석의 한계를 명확히 인식하기 위한 사전 규칙과 검증 계획을 문서화합니다.

#### 2. 세부 수행 절차

- **원본 목표·기본 코드·자료 분석 및 핵심 질문 검토:**
  - 관할 부대의 DOS 현황을 4×5 가상 격자 지도와 경보 색상으로 나타내고, 예측 경보마다 기여 인자를 설명하는 데 필요한 입력 데이터, 기간, 단위, 그리고 사용 가능한 시점의 제약 조건을 파악합니다.
  - 특정 입력 인자 하나의 값을 단순히 평균값으로 대체하여 예측 변화를 살피는 방식(국소 민감도 분석)이 지닌 통계적 한계와, 이를 SHAP이나 인과관계, 실제 결핍 원인으로 단정해선 안 되는 이유를 사전에 분석하고 정의합니다.
- **explain 입출력 계약 정의:**
  - 개별 행( row ) 데이터를 입력받아 예측값과 상위 3개의 입력 민감도(기여 인자)를 반환하는 핵심 함수 explain 의 입출력 계약 조건을 명확히 이해합니다.
- **AI 위임 범위 및 검증 계획 정의:**
  - AI 에이전트에 맡길 코드 구현 및 분석 범위, 데이터 무결성 검증 방법을 설정하고, 본인의 사전 판단과 검증 계획을 서식( forms/STR01_M06_forms.md…

#### 완료 확인:

- 이 활동의 확인 근거 M06 계획: 관할 부대의 DOS 현황을 가상 지도와 경보 색상으로 보여 주는 대시보드를 구현하고, 경보마다 예측에 기여한 인자를 설명한다.에 필요한 입력·기간·단위·사용 가능 시점과 explain의입출력 계약을 설명하고 본인의사전판단·AI위임·검증계획을 기록했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

#### 내가 기록할 것:

- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:

### 활동 2: AI와 코드 초안을 만들어 실행하기

**이번 활동의 과제:**

#### 1. 단계 개요

- **목표:**
  - 핵심 함수인 explain 을 구현·저장하고 첫 성공 실행 결과를 온전히 보존합니다.
  - 입력값인 개별 행( row ) 데이터를 바탕으로 예측값과 상위 3개의 입력 민감도(기여 인자)를 산출하며, 특정 인자의 값을 평균값으로 대체했을 때 나타나는 예측 변화의 절댓값을 비교하여 인자 중요도를 파악합니다.
  - 단, 이를 SHAP 분석, 인과관계, 혹은 실제 결핍의 원인으로 직접 단정하지 않도록 주의합니다.
  - 아울러 변경 과제(경보 사례 3건에서 상위 인자를 평균값으로 치환하고 변화 방향 직접 확인)를 수행하여 이미 결핍된 건의 규칙 설명과 예측 경보의 모델 설명을 명확히 구분합니다.
- **수행 방식:**
  - AI와 협업하여 XAI(설명 가능한 AI) 국소 민감도 분석 로직을 코드에 반영하고, 기준 코드 실행 및 변경 과제 실험을 수행하여 실제 코드와 출력 결과를 정확하게 확보합니다.

#### 2. 세부 수행 절차

+ **코드 구현 및 첫 실행**
  - 핵심 함수 explain 구현 및 첫 성공 실행: 개별 행 데이터를 받아 예측값과 상위 3개의 입력 민감도(기여 인자)를 반환하는 explain 함수를 구현합니다.
  - 입력 인자 하나의 값을 평균값으로 치환하여 예측값의 변화 절댓값을 비교하는 국소 민감도 방식을 적용하되, 이를 통계적 인과관계나 SHAP, 실제 결핍 원인으로 오인·단정하지 않도록 유의합니다.
  - 첫 성공 실행 결과, 로그, 표 및 그림 형태의 출력을 누락 없이 보존합니다.

+ **변경 과제와 전후 비교**
  - 변경 과제 수행 (경보 사례 3건 인자 치환 및 비교): 실제 경보 사례 3건을 선정하여 상위 기여 인자의 값을 평균값으로 치환한 뒤, 예측값이 어떻게 변동되는지 방향성을 직접 확인하고 기록합니다.
  - 이미 재고가 0이 되어 결핍된 건에 대한 규칙 기반 설명과, 향후 결핍을 예측하는 모델 기반 설명을 명확히 구분하여 적용합니다.
  - **실제 코드 및 출력 결과 저장:**
    - …

#### 완료 확인:

- 이 활동의 확인 근거 M06 구현: src/explainability/str01_xai_explainer.py의 explain를저장하고계약〔row → 예측값과 상위3개 입력 민감도.
- 평균값 대체로 예측 변화 절댓값을 비교한다.
- SHAP·인과·실제 결핍 원인으로 단정하지 않는다.〕을충족하는 첫성공실행·로그·표/그림을보존했는가?
- 이후Lesson의추가변경을실제코드에반영했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

#### 내가 기록할 것:

- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:

### 활동 3: 원자료와 결과를 대조·수정하기

**이번 활동의 과제:**

#### 1. 단계 개요

- **목표:**
  - 앞서 구현된 XAI 설명 모듈 코드를 저장하고 재실행하여, 이미 재고가 소진되어 0이 된 품목들도 가상 지도(4×5 격자)의 최악 경보에 올바르게 포함되는지 검증합니다.
  - 아울러 예측 경보 건당 기여도 상위 3개 인자의 자동 산출률 100%를 달성하고, 군수 장교와 군무원이 신뢰하고 조치할 수 있도록 설명의 명확성을 확보합니다.
  - 아울러 AI의 설명 및 코드 선택 3건 이상을 실제 군수 도메인 자료 및 실행 결과와 철저히 대조하여 보완된 답변을 도출합니다.
- **수행 방식:**
  - 단편적인 코드 실행에 그치지 않고, 100～102일의 3시점 재생 데이터와 실제 기간·분모·조건·수치를 바탕으로 최초 실행 결과와 수정 실행 결과를 비교 대조하며 AI의 답변을 비판적으로 교차 검증합니다.

#### 2. 세부 수행 절차

- **수정 코드 저장 및 재실행:**
  - XAI 설명 모듈 소스 코드( src/explainability/str01_xai_explainer.py )를 안전하게 저장하고 재실행하여 에러 여부와 실행 로그를 확보합니다.
- **가상 지도 및 재고 0인 품목 처리 검증:**
  - 4×5 격자의 가상 지도 상에서 이미 재고가 0이 된 품목이 최악 경보 상태로 정확히 표시되는지 확인합니다.
  - 상위 3개 인자 산출의 분모는 양수 재고의 예측 경보 건으로 한정하고, 이미 결핍된 건은 재고 관찰 규칙을 통해 올바르게 설명되고 있는지 점검합니다.
- **국소 민감도 해석의 한계 검증 및 자동 산출률 확인:**
  - 평균값 치환에 따른 예측 변화는 단순한 국소 민감도 분석 결과이므로 이를 SHAP이나 절대적 인과 효과로 오인하지 않는지 엄밀히 검토하고, 주요 예측 건당 기여도 상위 3개 인자 자동 산출률 100%를 달성했는지 확인합니다.
  - (단, 자동 산출률 100%가 설명 품질 검토 자체를 대신할 수는 없음을 인지합니다.) AI 설명·코드…

#### 완료 확인:

- **판정에 필요한 근거**
  - 이 활동의 확인 근거 M06 검증: [선택] 현황 지도·XAI 설명 — 주요 예측 건당 기여도 상위 3개 인자 자동 산출률 100%이며, 군수 장교와 군무원이 신뢰하고 조치할 수 있게 설명이 명확한가.
  - 실제기간·분모·조건·수치로판정하고 최초/수정실행을비교했는가?
- **미션별 확인 사항**
  - **수행 조건과 판정 기준 · 판정 상태와 적용 범위**
    - 가상 지도는 4×5 격자이며 실제 위치가 아닙니다.
    - 100～102일의 3시점 재생입니다.
    - 이미 재고 0인 품목은 최악 경보에 포함합니다.
  - **수행 조건과 판정 기준 · 측정 조건과 결과 해석**
    - 상위 3인자 산출률 분모는 양수 재고의 예측 경보이고, 이미 결핍은 재고 관찰 규칙으로 설명합니다.
    - 평균 치환의 예측 변화는 국소 민감도이며 SHAP·인과 효과가 아닙니다.
    - 자동 산출 100%는 설명 품질 검토를 대신하지 않습니다.
- **AI 검토 기록**
  - AI 주장/선택3건을근거로검토·보완·재실행했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서…

#### 내가 기록할 것:

- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:

### 활동 4: 최종 결과와 수행 상태 기록하기

**이번 활동의 과제:**

#### 1. 단계 개요

- **목표:**
  - 미션 6(군단/사단 관할 부대 보급 수준(DOS) 실시간 현황 지도 및 경보 대시보드 구현) 전 과정을 수행하며 산출된 XAI 설명 모듈 소스 코드 , 모델 분석 보고서 , 예측 결과 데이터 파일 , 그리고 실습 기록 서식 을 누락 없이 집대성합니다.
  - 아울러 정량적 목표 달성 여부, 미측정 영역, 문서 작성 상태 및 교관(또는 시스템) 승인 대기 상태를 명확히 구분하여 최종 판단을 기록합니다.
- **수행 방식:**
  - 단편적인 파일 제출에 치중하지 않고, 이번 미션에서 구축한 가상 지도(4×5 격자) 대시보드와 XAI 설명 결과(주요 예측 건당 기여도 상위 3개 인자 자동 산출률 100% 달성 여부 등)를 객관적으로 검수하여 후속 미션(미션 7: 결핍 부대 긴급 물자 전환 결정을 위한 군수처장 HITL 승인 워크플로우 설계)으로 온전히 인계할 수 있도록 패키지를 정비합니다.

#### 2. 세부 수행 절차

- **핵심 산출 파일 패키지 확정 및 제출:**
  - XAI 설명 모듈 소스 코드인 src/explainability/str01_xai_explainer.py , 모델 분석 보고서인 reports/str01_xai_report.md , 처리 결과 데이터인 outputs/M06/results.json , 그리고 실습 기록 서식인 forms/STR01_M06_forms.md 의 실제 작성 파일들을 최종 점검하고 제출 상태로 정리합니다.
- **코드 실행 결과 및 AI 대화 기록 연계 보존:**
  - explain 함수 구현 및 경보 사례 인자 치환 실험 결과, 그리고 분석 과정에서 AI 에이전트와 나눈 핵심 질의응답 및 코드 선택 근거 대화 내용을 실습 대화 보존 위치에 명확히 연결합니다.
- **상태별 최종 판단 기록 작성:**
  - 정량적 목표 달성 여부, 현장 여건상 미처 측정하지 못한 영역, 문서 작성 완료 상태, 그리고 실…

#### 완료 확인:

- 이 활동의 확인 근거 M06 완료: src/explainability/str01_xai_explainer.py, reports/str01_xai_report.md, outputs/M06/results.json, forms/STR01_M06_forms.md의실제작성·최종값·대화보존위치·선택이유가연결되는가?
- 미달·미측정·문서미작성·실제승인대기를정확히표시했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

#### 내가 기록할 것:

- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:


## 🔍 교관 평가 기준

- [정량적 평가] [선택] 현황 지도·XAI 설명 — 주요 예측 건당 기여도 상위 3개 인자 자동 산출률 100%이며, 군수 장교와 군무원이 신뢰하고 조치할 수 있게 설명이 명확한가. PASS 증거 제공된조건·단위·기간·분모와실제수치/건수/항목근거를연결합니다. 목표미달·0기준·미측정은그대로표시합니다. 보완 방법 기준별빠진항목과미달원인을찾아수정하고재검증합니다.
- **[정성적 평가] AI 활용: 본인 계획·실제 대화·코드 변경·검토3건·보완·재실행·전후 비교·최종 선택.**
  - **PASS 증거**
    - **수행 조건과 판정 기준 · 판정 상태와 적용 범위**
      - 가상 지도는 4×5 격자이며 실제 위치가 아닙니다.
      - 100～102일의 3시점 재생입니다.
      - 이미 재고 0인 품목은 최악 경보에 포함합니다.
    - **수행 조건과 판정 기준 · 측정 조건과 결과 해석**
      - 상위 3인자 산출률 분모는 양수 재고의 예측 경보이고, 이미 결핍은 재고 관찰 규칙으로 설명합니다.
      - 평균 치환의 예측 변화는 국소 민감도이며 SHAP·인과 효과가 아닙니다.
      - 자동 산출 100%는 설명 품질 검토를 대신하지 않습니다.
    - **보완 방법**
      - 보완 방법 근거없는원인·규정·완료추정은수정하거나확인필요로보류합니다.
- **[진행 추적 및 완료 기준] 정성·검증:**
  - **수행 조건과 판정 기준 · 판정 상태와 적용 범위**
    - 가상 지도는 4×5 격자이며 실제 위치가 아닙니다.
    - 100～102일의 3시점 재생입니다.
    - 이미 재고 0인 품목은 최악 경보에 포함합니다.
  - **수행 조건과 판정 기준 · 측정 조건과 결과 해석**
    - 상위 3인자 산출률 분모는 양수 재고의 예측 경보이고, 이미 결핍은 재고 관찰 규칙으로 설명합니다.
    - 평균 치환의 예측 변화는 국소 민감도이며 SHAP·인과 효과가 아닙니다.
    - 자동 산출 100%는 설명 품질 검토를 대신하지 않습니다.
  - **제출 파일 및 완료 기록**
    - 파일: src/explainability/str01_xai_explainer.py, reports/str01_xai_report.md, outputs/M06/results.json, forms/STR01_M06_forms.md 실제AI요청·답변·근거대조3건이상·보완·재확인·본인최종선택과작성/실행기록이연결됩니다.
    - 최종ZIP생성은실제학습완료·교관승인 인증이아닙니다.
  - **보완 방법**
    - 보완 방법 미작성·미실행·검토누락을보완하고교관검토상태를별도로표시합니다.
실제 수행·실행·사람의 승인 여부를 별도로 판정하세요.

**교관용 결과 저장:** 마지막 통합 화면에서 산출물을 개별 작성·수정·저장하고, 학습기록·실제 AI 대화·근거·3건 검토·최종 상태를 입력한 후 ZIP으로 저장할 수 있습니다.

교관용 참고 예시와 실제 실행 증거는 구분하세요.


In [ ]:
# @title ⑤ 교관용 개별 산출물·학습기록 통합 입력, 저장 및 ZIP 다운로드 (실행 버튼을 누르세요) { display-mode: "form" }
# 입력 폼이 나타나면 소스코드 대신 버튼을 이용하세요. Colab/Jupyter에서 ipywidgets 필요.
from pathlib import Path

if not (globals().get('STR01_PROJECT_READY', False) and globals().get('STR01_PROJECT_DIR') and Path(STR01_PROJECT_DIR).is_dir()):
    print('먼저 ① 실습자료 자동 준비 및 환경 설정 셀을 실행한 뒤 이 셀을 다시 실행하세요. ZIP 업로드는 필요 없습니다.')
else:
    def _load_notebook_widgets():
        from importlib import import_module, invalidate_caches
        import subprocess
        import sys

        try:
            return import_module('ipywidgets')
        except ImportError:
            print('이 노트북 커널에 ipywidgets가 없어 설치합니다.')

        try:
            subprocess.run(
                [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                 '--retries', '0', '--timeout', '10', 'ipywidgets'],
                check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=60,
            )
            invalidate_caches()
            return import_module('ipywidgets')
        except (subprocess.SubprocessError, OSError, ImportError):
            print('이 커널에서 ipywidgets를 설치하거나 불러오지 못했습니다.')
            print('설치 가능한 환경에서는 새 셀에서 %pip install ipywidgets 를 실행한 뒤 이 셀을 다시 실행하세요.')
            print('설치할 수 없는 환경에서는 제공된 Markdown 작성 양식을 사용하세요.')
            return None

    widgets = _load_notebook_widgets()
    if widgets is not None:
        import json, csv, io, ast, zipfile, datetime, pathlib
        from pathlib import Path
        import os
        from urllib.parse import quote
        from IPython.display import HTML, display, Markdown, clear_output

        MISSION_CODE = 6
        DELIVERABLE_PATHS = ['src/explainability/str01_xai_explainer.py', 'reports/str01_xai_report.md', 'outputs/M06/results.json', 'forms/STR01_M06_forms.md']
        PROJECT_NAME = 'STR01'
        NOTEBOOK_ROLE = '교관용 Solution'
        WORK_ROOT = Path(STR01_PROJECT_DIR).resolve()

        RECORD_DIR = WORK_ROOT / f'STR01_M{MISSION_CODE:02d}_학습기록'
        RECORD_DIR.mkdir(parents=True, exist_ok=True)
        ZIP_FILENAME = f'STR01_m{MISSION_CODE:02d}_review.zip'

        # 반드시 CMS 지정 경로만 사용하며 상대 경로의 작업공간 이탈을 방지합니다.
        def safe_path(relative):
            target = (WORK_ROOT / relative).resolve()
            if not target.is_relative_to(WORK_ROOT):
                raise ValueError('작업공간 밖의 경로는 사용할 수 없습니다.')
            return target

        def valid_text(path, content):
            suffix = Path(path).suffix.lower()
            if suffix == '.json':
                json.loads(content)
            elif suffix == '.ipynb':
                doc = json.loads(content)
                if not isinstance(doc, dict) or not isinstance(doc.get('cells'), list) or 'nbformat' not in doc:
                    raise ValueError('노트북 JSON에 cells 또는 nbformat 정보가 없습니다.')
            elif suffix == '.py':
                ast.parse(content)
            elif suffix == '.csv':
                if not content.strip():
                    raise ValueError('CSV 데이터가 비어 있습니다.')
                rows = list(csv.reader(io.StringIO(content)))
                if not rows or not rows[0] or not any(x.strip() for x in rows[0]):
                    raise ValueError('CSV 첫 행에 열 이름을 입력하세요.')
                if any(len(row) != len(rows[0]) for row in rows if row):
                    raise ValueError('CSV 열 개수가 행마다 다릅니다.')

        # 산출물 입력 화면 — 파일별 별도 탭
        editors = {}
        file_tabs = []
        file_buttons = {}
        for relative in DELIVERABLE_PATHS:
            suffix = Path(relative).suffix.lower()
            hint = {
                '.md': '보고서·양식의 제목, 분석 결과, 근거와 수정 이유를 Markdown으로 작성합니다.',
                '.csv': '첫 줄에 열 이름, 다음 줄부터 데이터 행을 쉼표로 구분하여 입력합니다.',
                '.json': '키와 값이 있는 유효한 JSON 형식으로 실제 결과를 입력합니다.',
                '.py': '직접 구현하고 확인한 Python 소스코드를 입력합니다.',
                '.ipynb': '노트북 JSON 원문을 입력하거나 기존 파일에서 불러옵니다. JSON 노트북 형식을 유지하세요.'
            }.get(suffix, '실제 작성한 내용을 입력합니다.')
            editor = widgets.Textarea(value='', placeholder='여기에 내용을 작성하세요. 입력하지 않은 파일은 저장하지 않습니다.',
                                      layout=widgets.Layout(width='100%', height='250px'))
            status = widgets.HTML('<span style="color:#777">아직 저장하지 않았습니다.</span>')
            load_btn = widgets.Button(description='기존 파일 불러오기', icon='folder-open', button_style='')
            save_btn = widgets.Button(description='이 파일 저장', icon='save', button_style='success')
            editors[relative] = {'editor': editor, 'status': status, 'loaded': False, 'saved': False, 'dirty': False, 'saved_value': ''}
            def on_edit(change, key=relative):
                state = editors[key]
                state['dirty'] = change['new'] != state['saved_value']
                state['saved'] = state['loaded'] and not state['dirty']
                if state['dirty']:
                    state['status'].value = '<span style="color:#a65">● 수정됨 — 저장 버튼을 누르세요.</span>'
                elif state['saved']:
                    state['status'].value = '<span style="color:#286">저장된 내용과 같습니다.</span>'
                else:
                    state['status'].value = '<span style="color:#777">아직 저장하지 않았습니다.</span>'
            editor.observe(on_edit, names='value')
            def load_clicked(button, key=relative):
                state = editors[key]
                target = safe_path(key)
                if not target.is_file():
                    state['status'].value = '<span style="color:#a22">파일이 아직 없습니다. 직접 입력 후 저장하세요.</span>'
                    return
                if state['dirty']:
                    state['status'].value = '<span style="color:#a22">미저장 내용이 있습니다. 불러오기 전에 저장하세요.</span>'
                    return
                try:
                    state['loaded'] = False
                    state['saved_value'] = target.read_text(encoding='utf-8')
                    state['editor'].value = state['saved_value']
                    state['loaded'] = True
                    state['dirty'] = False
                    state['saved'] = True
                    state['status'].value = '<span style="color:#286">파일을 불러왔습니다. 변경하면 다시 저장하세요.</span>'
                except Exception as exc:
                    state['status'].value = '<span style="color:#a22">불러오기 오류: '+str(exc)+'</span>'
            def save_clicked(button, key=relative):
                save_one(key)
            load_btn.on_click(load_clicked)
            save_btn.on_click(save_clicked)
            file_buttons[relative] = {'load':load_btn,'save':save_btn}
            file_tabs.append(widgets.VBox([
                widgets.HTML('<b>저장 경로:</b> <code>'+relative+'</code><br><span>'+hint+'</span><br><span style="color:#a55">기존 파일이 있다면 저장 시 덮어씁니다. 실습 완료·승인과는 다릅니다.</span>'),
                editor, widgets.HBox([load_btn, save_btn]), status
            ]))

        def save_one(relative):
            state = editors[relative]
            value = state['editor'].value
            if not value.strip():
                state['status'].value = '<span style="color:#a22">비어 있어 저장하지 않았습니다.</span>'
                return False
            try:
                valid_text(relative, value)
                target = safe_path(relative)
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_text(value, encoding='utf-8')
                state['saved_value'] = value
                (WORK_ROOT / ZIP_FILENAME).unlink(missing_ok=True)
                state['loaded'] = True
                state['dirty'] = False
                state['saved'] = True
                state['status'].value = '<span style="color:green">✓ 저장 완료: '+relative+'</span>'
                return True
            except Exception as exc:
                state['status'].value = '<span style="color:#a22">저장 실패: '+str(exc)+'</span>'
                return False

        # 학습기록 및 AI 원문, 자료 대조, 검토 3건 이상
        record_specs = [
         ('plan','학습 계획·수행 기록','이번 미션에서 직접 수행한 내용, 실행 조건 및 최초 결과를 작성하세요.'),
         ('ai_request','실제 AI 질문 원문','실제로 입력한 질문 원문만 작성하세요. 사용하지 않았다면 비워 둡니다.'),
         ('ai_response','실제 받은 AI 답변 원문','실제로 받은 답변 전체를 붙여 넣으세요. 임의로 생성한 대화는 기록하지 않습니다.'),
         ('evidence','실제 출력·자료 대조 근거','실제 출력값, 원자료 위치, 전후 비교 및 계산 근거를 적으세요.'),
         ('revision','검토 및 수정 내용 종합','검토 3건의 발견사항과 코드를 수정·재실행한 기록을 종합하세요.'),
         ('decision','최종 판단과 미달·미측정·미실시 사유','달성 여부와 남은 부분, 근거 및 추가 확인사항을 적으세요.'),
        ]
        record_widgets = {}
        record_items = []
        for key, title, hint in record_specs:
            field = widgets.Textarea(placeholder=hint, layout=widgets.Layout(width='100%',height='115px'))
            record_widgets[key] = field
            record_items.append(widgets.VBox([widgets.HTML('<b>'+title+'</b><br><small>'+hint+'</small>'),field]))

        reviews = []
        for i in range(3):
            claim = widgets.Textarea(placeholder=f'{i+1}번째 AI 주장·코드 선택 또는 점검 대상',layout=widgets.Layout(width='100%',height='65px'))
            ground = widgets.Textarea(placeholder='원자료·실행 출력의 위치 및 비교한 근거',layout=widgets.Layout(width='100%',height='65px'))
            action = widgets.Textarea(placeholder='맞음·수정·보류, 수정 내용 및 재확인 결과',layout=widgets.Layout(width='100%',height='65px'))
            reviews.append((claim,ground,action))
            record_items.append(widgets.VBox([widgets.HTML(f'<b>검토 {i+1} (최소 3건 중 {i+1}번째)</b>'),claim,ground,action]))

        status_check = widgets.SelectMultiple(options=['달성','미달','미측정','미실시','추가 확인 필요'],
                                              value=(),description='수행 상태',layout=widgets.Layout(width='70%',height='115px'))
        approval = widgets.Dropdown(options=['미확인 / 교관 승인 대기','실제 교관 검토 완료 (기록 확인 필요)'],
                                    value='미확인 / 교관 승인 대기', description='교관 상태',layout=widgets.Layout(width='85%'))
        record_notice = widgets.HTML('<span style="color:#666">입력 후 [학습기록 저장]을 누르세요. 미작성·미측정은 그대로 표시합니다.</span>')
        record_dirty = {'value': False}
        def _mark_record_dirty(change):
            record_dirty['value'] = True
            record_notice.value = '<span style="color:#a65">● 학습기록에 저장하지 않은 변경이 있습니다.</span>'
        for field in list(record_widgets.values()) + [v for group in reviews for v in group] + [status_check,approval]:
            field.observe(_mark_record_dirty,names='value')

        # 앞서 작성한 학습기록을 자동으로 불러와 이어서 수정할 수 있습니다.
        previous_audit = RECORD_DIR/'learner_audit.json'
        if previous_audit.is_file():
            try:
                previous = json.loads(previous_audit.read_text(encoding='utf-8'))
                for key, field in record_widgets.items():
                    field.value = str(previous.get(key,'') or '')
                for group, past in zip(reviews, previous.get('reviews',[])):
                    for widget, key in zip(group,('claim','evidence','revision_and_recheck')):
                        widget.value = str(past.get(key,'') or '')
                status_check.value = tuple(x for x in previous.get('status',[]) if x in status_check.options)
                if previous.get('instructor_approval') in approval.options:
                    approval.value = previous['instructor_approval']
                record_dirty['value'] = False
                record_notice.value = '<span style="color:#286">이전 학습기록을 불러왔습니다. 수정 후 다시 저장하세요.</span>'
            except Exception as exc:
                record_notice.value = '<span style="color:#a22">기존 학습기록 읽기 오류: '+str(exc)+'</span>'

        def audit_object():
            return {
                'project': PROJECT_NAME, 'mission': MISSION_CODE, 'notebook_role': NOTEBOOK_ROLE,
                'plan':record_widgets['plan'].value, 'ai_request':record_widgets['ai_request'].value,
                'ai_response':record_widgets['ai_response'].value, 'evidence':record_widgets['evidence'].value,
                'revision':record_widgets['revision'].value, 'decision':record_widgets['decision'].value,
                'reviews':[{'claim':a.value,'evidence':b.value,'revision_and_recheck':c.value} for a,b,c in reviews],
                'status':list(status_check.value),'instructor_approval':approval.value,
                'saved_at':datetime.datetime.now().isoformat(timespec='seconds')
            }

        def save_audit(button=None):
            try:
                record = audit_object()
                (RECORD_DIR/'learner_audit.json').write_text(json.dumps(record,ensure_ascii=False,indent=2),encoding='utf-8')
                (RECORD_DIR/'AI_대화원문.txt').write_text('실제 AI 질문 원문\n'+record['ai_request']+'\n\n실제 AI 답변 원문\n'+record['ai_response'],encoding='utf-8')
                sections = [f'# STR01 미션 {MISSION_CODE:02d} 학습기록']
                for name,key in [('학습 계획·수행 기록','plan'),('실제 출력·자료 대조 근거','evidence'),('검토 및 수정 내용','revision'),('최종 판단','decision')]:
                    sections.append('## '+name+'\n'+(record[key] or '(미작성)'))
                for i,item in enumerate(record['reviews'],1):
                    sections.append(f"## 검토 {i}\n- 검토 대상: {item['claim'] or '(미작성)'}\n- 대조 근거: {item['evidence'] or '(미작성)'}\n- 수정 및 재확인: {item['revision_and_recheck'] or '(미작성)'}")
                sections.append('## 상태\n'+(', '.join(record['status']) or '미선택')+'\n\n교관 상태: '+record['instructor_approval'])
                (RECORD_DIR/'학습기록.md').write_text('\n\n'.join(sections)+'\n',encoding='utf-8')
                record_dirty['value'] = False
                (WORK_ROOT / ZIP_FILENAME).unlink(missing_ok=True)
                completed = sum(all(v.strip() for v in [a.value,b.value,c.value]) for a,b,c in reviews)
                record_notice.value = f'<span style="color:green">✓ 학습기록 저장 완료. 근거가 모두 입력된 검토: {completed}/3건. 저장은 PASS나 교관 승인과 다릅니다.</span>'
                return True
            except Exception as exc:
                record_notice.value = '<span style="color:#a22">학습기록 저장 실패: '+str(exc)+'</span>'
                return False

        save_audit_btn = widgets.Button(description='학습기록 저장',icon='save',button_style='success')
        save_audit_btn.on_click(save_audit)
        record_tab = widgets.VBox(record_items+[widgets.HTML('<b>최종 수행 상태</b> — 해당되는 항목을 선택하세요. 선택만으로 자동 평가되지 않습니다.'),status_check,approval,save_audit_btn,record_notice])

        tabs = widgets.Tab(children=file_tabs+[record_tab])
        for i, path in enumerate(DELIVERABLE_PATHS):
            tabs.set_title(i, f'산출물 {i+1}')
        tabs.set_title(len(file_tabs),'학습기록')

        summary_output = widgets.Output()
        all_btn = widgets.Button(description='작성한 산출물 전체 저장', icon='save',button_style='info')
        zip_btn = widgets.Button(description='저장 파일 ZIP 생성',icon='archive',button_style='warning')
        dl_btn = widgets.Button(description='ZIP 다운로드',icon='download',button_style='')

        def save_all(button):
            ok = 0
            for rel in DELIVERABLE_PATHS:
                if editors[rel]['editor'].value.strip() and save_one(rel):
                    ok += 1
            with summary_output:
                summary_output.clear_output()
                print(f'산출물 {ok}개 저장 완료. 파일별 상태를 확인하세요. 학습기록은 별도 [학습기록 저장] 버튼을 누르세요.')

        def build_zip(button):
            with summary_output:
                summary_output.clear_output()
                if record_dirty['value']:
                    print('❌ 학습기록에 미저장 변경 내용이 있습니다. [학습기록 저장]을 먼저 누르세요.')
                    return
                dirty = [p for p,s in editors.items() if s['dirty']]
                if dirty:
                    print('❌ 미저장 변경 내용이 있습니다. 아래 파일들을 먼저 저장하세요:')
                    for p in dirty: print(' -',p)
                    return
                # 기존에 제공된 파일은 ZIP에 임의로 포함하지 않고 현재 화면에서 불러오거나 저장한 파일만 수집합니다.
                saved = [p for p,s in editors.items() if s['saved'] and safe_path(p).is_file()]
                audit_files = [p for p in RECORD_DIR.rglob('*') if p.is_file()]
                if not saved and not audit_files:
                    print('❌ 저장된 결과물이 없습니다. 산출물 또는 학습기록을 저장하세요.')
                    return
                dest = WORK_ROOT / ZIP_FILENAME
                with zipfile.ZipFile(dest,'w',compression=zipfile.ZIP_DEFLATED) as archive:
                    for rel in saved: archive.write(safe_path(rel), arcname=rel)
                    for f in audit_files: archive.write(f, arcname=str(f.relative_to(WORK_ROOT)))
                print('✅ ZIP 생성:', dest)
                print('산출물:',len(saved),'개 / 학습기록:',len(audit_files),'개')
                print('※ ZIP 생성은 학습 완료 또는 교관 승인을 의미하지 않습니다.')

        def download_zip(button):
            with summary_output:
                summary_output.clear_output()
                target = WORK_ROOT / ZIP_FILENAME
                if record_dirty['value'] or any(s['dirty'] for s in editors.values()):
                    print('미저장 변경이 있습니다. 저장 후 ZIP을 다시 생성해 주세요.')
                    return
                if not target.is_file():
                    print('먼저 [저장 파일 ZIP 생성]을 눌러 주세요.')
                    return
                try:
                    from google.colab import files as colab_files
                    colab_files.download(str(target))
                    print('다운로드를 요청했습니다.')
                except ImportError:
                    start_dir = globals().get('notebook_directory', WORK_ROOT.parent)
                    href = quote(Path(os.path.relpath(target, start_dir)).as_posix())
                    display(HTML(f'<a href="{href}" download>ZIP 다운로드: {target.name}</a>'))

        all_btn.on_click(save_all)
        zip_btn.on_click(build_zip)
        dl_btn.on_click(download_zip)

        print(f'✅ STR01 미션 {MISSION_CODE:02d} 산출물·학습기록 입력 화면')
        display(Markdown('**사용 순서:** ① 각 산출물 탭에서 작성하거나 기존 파일 불러오기 → ② 파일별 저장 → ③ 학습기록 탭에서 실제 AI 원문·근거·검토 3건·최종 상태 입력 및 저장 → ④ ZIP 생성 → ⑤ 다운로드.'))
        display(Markdown('**주의:** 원본 실습의 TODO는 직접 실행해야 합니다. 교관용 참고 출력과 실제 수행 기록은 구분하세요. 입력창은 실제 수행 내용을 기록하기 위한 기능이며, 미작성 결과를 자동으로 완성하지 않습니다.'))
        display(tabs,widgets.HBox([all_btn,zip_btn,dl_btn]),summary_output)
